In [0]:
fetch_date = dbutils.widgets.get("fetch_date")
source_table = dbutils.widgets.get("source_table")
landing_table = dbutils.widgets.get("landing_table")
office_table = dbutils.widgets.get("office_table")
client_table = dbutils.widgets.get("client_table")

In [0]:
display(
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW expectednetrevenue_src AS
SELECT 
  CAST(ReportingDate AS DATE) AS ReportingDate,
  CAST(FacilityCode AS INT) AS FacilityCode,
  CAST(AcctNbr AS STRING) AS AcctNbr,
  CAST(NetRevenue AS DOUBLE) AS ExpectedNetRevenue,
  CAST(SourceSystemKey AS INT) AS SourceSystemKey,
  current_timestamp() AS _load_timestamp
FROM (
  WITH
  obd AS (
    SELECT
        office_key,
        client_key,
        invoice_number,
        net_revenue,
        date_entered_key
    FROM {source_table}
    WHERE account_balance <> 0
    AND date_entered_key = date_format(DATE('{fetch_date}'), 'yyyyMMdd')
  ),
  expectednetrevenue_cte AS (
    SELECT
        to_date(CAST(date_entered_key AS STRING), 'yyyyMMdd') AS ReportingDate,
        ofc.OfficeNumber AS FacilityCode,
        CASE
            WHEN UPPER(obd.invoice_number) = 'ADV'
                THEN concat('ADV', ' - ', clt.SourceSystemId)
            ELSE obd.invoice_number
        END AS AcctNbr,
        obd.net_revenue AS NetRevenue,
        0 AS SourceSystemKey
    FROM obd
    LEFT JOIN {office_table} ofc
        ON obd.office_key = ofc.OfficeKey
    LEFT JOIN {client_table} clt
        ON clt.ClientKey = obd.client_key
  ),
  expectednetrevenue_clean AS (
    SELECT *,
    ROW_NUMBER() OVER (
          PARTITION BY
            AcctNbr, FacilityCode
          ORDER BY AcctNbr
      ) AS rn
    FROM expectednetrevenue_cte
  )
  SELECT ReportingDate, FacilityCode, AcctNbr, NetRevenue, SourceSystemKey
  FROM expectednetrevenue_clean
  WHERE rn=1
) AS src
""")
)

In [0]:
display(
spark.sql(f"""
MERGE INTO {landing_table} tgt
USING expectednetrevenue_src src
    ON tgt.AcctNbr = src.AcctNbr
    AND tgt.ReportingDate = src.ReportingDate
    AND tgt.SourceSystemKey = 0

WHEN MATCHED THEN
UPDATE SET
    tgt.FacilityCode = src.FacilityCode,
    tgt.ExpectedNetRevenue = src.ExpectedNetRevenue,
    tgt._load_timestamp = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
    ReportingDate,
    FacilityCode,
    AcctNbr,
    ExpectedNetRevenue,
    SourceSystemKey,
    _load_timestamp
)
VALUES (
    src.ReportingDate,
    src.FacilityCode,
    src.AcctNbr,
    src.ExpectedNetRevenue,
    src.SourceSystemKey,
    current_timestamp()
)
""")
)